# Normalización de la tabla origen

In [16]:
import pandas as pd
import numpy as np

In [17]:
data = pd.read_csv(r"C:\Users\carlo\OneDrive\Escritorio\Proyecto Rotación IBM\Vertiente de BI\WA_Fn-UseC_-HR-Employee-Attrition (1).csv")
data.head() #Para verificar la correcta carga del dataset

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [18]:
# Para verificar completitud de los datos antes de comenzar a intepretar
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [19]:
# ya que se verificó que los datos están completos, se puede comenzar a modelar los datos.-

data.head().T #formato para visualizar de mejor manera la data

,0,1,2,3,4
Age,41,49,37,33,27
Attrition,Yes,No,Yes,No,No
BusinessTravel,Travel_Rarely,Travel_Frequently,Travel_Rarely,Travel_Frequently,Travel_Rarely
DailyRate,1102,279,1373,1392,591
Department,Sales,Research & Development,Research & Development,Research & Development,Research & Development
DistanceFromHome,1,8,2,3,2
Education,2,1,2,4,1
EducationField,Life Sciences,Life Sciences,Other,Life Sciences,Medical
EmployeeCount,1,1,1,1,1
EmployeeNumber,1,2,4,5,7


In [20]:
# Resumen para conocer la naturaleza de las variables
resumen_naturaleza = pd.DataFrame({
    'Tipo de Dato': data.dtypes,
    'Valores únicos': data.nunique(),
    'Ejemplos': [data[col].unique()[:6] for col in data.columns] # Muestra los primeros 5 valores encontrados (en caso de que haya más de 5)
})

# Visuaizando el resumen
resumen_naturaleza


,Tipo de Dato,Valores únicos,Ejemplos
Age,int64,43,"[41, 49, 37, 33, 27, 32]"
Attrition,object,2,"[Yes, No]"
BusinessTravel,object,3,"[Travel_Rarely, Travel_Frequently, Non-Travel]"
DailyRate,int64,886,"[1102, 279, 1373, 1392, 591, 1005]"
Department,object,3,"[Sales, Research & Development, Human Resources]"
DistanceFromHome,int64,29,"[1, 8, 2, 3, 24, 23]"
Education,int64,5,"[2, 1, 4, 3, 5]"
EducationField,object,6,"[Life Sciences, Other, Medical, Marketing, Tec..."
EmployeeCount,int64,1,[1]
EmployeeNumber,int64,1470,"[1, 2, 4, 5, 7, 8]"


In [21]:
data.describe(include="all").T #para conocer estadísticas básicas.

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Age,1470.0,NaN,NaN,NaN,36.92381,9.135373,18.0,30.0,36.0,43.0,60.0
Attrition,1470,2,No,1233,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BusinessTravel,1470,3,Travel_Rarely,1043,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DailyRate,1470.0,NaN,NaN,NaN,802.485714,403.5091,102.0,465.0,802.0,1157.0,1499.0
Department,1470,3,Research & Development,961,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DistanceFromHome,1470.0,NaN,NaN,NaN,9.192517,8.106864,1.0,2.0,7.0,14.0,29.0
Education,1470.0,NaN,NaN,NaN,2.912925,1.024165,1.0,2.0,3.0,4.0,5.0
EducationField,1470,6,Life Sciences,606,NaN,NaN,NaN,NaN,NaN,NaN,NaN
EmployeeCount,1470.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
EmployeeNumber,1470.0,NaN,NaN,NaN,1024.865306,602.024335,1.0,491.25,1020.5,1555.75,2068.0


--------------------------------------------------------------

# Normalización de la tabla origen

### Conociendo la naturaleza de cada variable se puede proceder a crear cada una de las tablas/entidades que alimentarán la base de datos.

-------------------------------------------------------

## 1. Tabla Employee

### Justificación

Esta tabla representa la identidad y características demográficas del empleado. Incluye atributos que describen quién es la persona, no su trabajo ni su desempeño.

- Son variables estáticas (no cambian frecuentemente).

- Son atributos personales, no laborales.

- Son la base para relacionar todas las demás tablas mediante EmployeeNumber.

Employee es la tabla central del modelo.


### Variables:

EmployeeNumber

Age

Gender

MaritalStatus

DistanceFromHome

Education

EducationField

In [22]:
Employee = pd.DataFrame(data[[
    'EmployeeNumber',
    'Age',
    'Gender',
    'MaritalStatus',
    'DistanceFromHome',
    'Education',
    'EducationField'
]])

employee.head()

,EmployeeNumber,Age,Gender,MaritalStatus,DistanceFromHome,Education,EducationField
0,1,41,Female,Single,1,2,Life Sciences
1,2,49,Male,Married,8,1,Life Sciences
2,4,37,Male,Single,2,2,Other
3,5,33,Female,Married,3,4,Life Sciences
4,7,27,Male,Married,2,1,Medical


## 2. Tabla JobInfo

### Justificación 

Esta tabla contiene la información estructural del puesto del empleado.

- Describe qué trabajo realiza.

- Describe en qué área está.

- Describe su nivel jerárquico.

- Incluye condiciones laborales como viajes y horas extra.

Estas variables pertenecen al puesto, no a la persona.
Por eso se separan de Employee.

### Variables:

EmployeeNumber (FK)

Department

JobRole

JobLevel

BusinessTravel

OverTime

In [23]:
JobInfo = pd.DataFrame(data[[
    'EmployeeNumber',
    'Department',
    'JobRole',
    'JobLevel',
    'BusinessTravel',
    'OverTime'
]])

jobinfo.head()

,EmployeeNumber,Department,JobRole,JobLevel,BusinessTravel,OverTime
0,1,Sales,Sales Executive,2,Travel_Rarely,Yes
1,2,Research & Development,Research Scientist,2,Travel_Frequently,No
2,4,Research & Development,Laboratory Technician,1,Travel_Rarely,Yes
3,5,Research & Development,Research Scientist,1,Travel_Frequently,Yes
4,7,Research & Development,Laboratory Technician,1,Travel_Rarely,No


## 3. Tabla SatisfactionScores

### Justificación

Esta tabla agrupa todas las variables ordinales relacionadas con la percepción del empleado:

- Satisfacción con el ambiente

- Satisfacción con el trabajo

- Satisfacción con relaciones

- Involucramiento

- Balance vida–trabajo

Estas variables son subjetivas, son medidas internas, son encuestas.

No pertenecen al puesto ni a la compensación, por eso se agrupan en una tabla independiente.

### Variables

EmployeeNumber (FK)

EnvironmentSatisfaction

JobSatisfaction

RelationshipSatisfaction

JobInvolvement

WorkLifeBalance

In [24]:
SatisfactionScores = pd.DataFrame(data[[
    'EmployeeNumber',
    'EnvironmentSatisfaction',
    'JobSatisfaction',
    'RelationshipSatisfaction',
    'JobInvolvement',
    'WorkLifeBalance'
]])

SatisfactionScores.head()

,EmployeeNumber,EnvironmentSatisfaction,JobSatisfaction,RelationshipSatisfaction,JobInvolvement,WorkLifeBalance
0,1,2,4,1,3,1
1,2,3,2,4,2,3
2,4,4,3,2,2,3
3,5,4,3,3,3,3
4,7,1,2,4,3,3


## 4. Tabla Compensation

### Justificación

Esta tabla contiene toda la información relacionada con compensación económica:

- Sueldo mensual

- Sueldo por hora

- Incrementos salariales

- Opciones de acciones

Estas variables son financieras, cambian con el tiempo, no pertenecen a la identidad ni al puesto.

Son métricas clave para BI, por eso se separan en una tabla especializada.

### Variables

EmployeeNumber (FK)

MonthlyIncome

MonthlyRate

DailyRate

HourlyRate

PercentSalaryHike

StockOptionLevel

In [25]:
Compensation = pd.DataFrame(data[[
    'EmployeeNumber',
    'MonthlyIncome',
    'MonthlyRate',
    'DailyRate', 
    'HourlyRate',
    'PercentSalaryHike',
    'StockOptionLevel'
]])

Compensation.head()

,EmployeeNumber,MonthlyIncome,MonthlyRate,DailyRate,HourlyRate,PercentSalaryHike,StockOptionLevel
0,1,5993,19479,1102,94,11,0
1,2,5130,24907,279,61,23,1
2,4,2090,2396,1373,92,15,0
3,5,2909,23159,1392,56,11,0
4,7,3468,16632,591,40,12,1


## 5. Tabla PerformanceHistory

### Justificación

Esta tabla representa la trayectoria laboral del empleado:

- Años de experiencia

- Años en la empresa

- Años en el rol actual

- Años desde último ascenso

- Años con el gerente actual

- Número de empresas previas

- Entrenamientos recibidos

- Calificación de desempeño

Estas variables son históricas, cambian con el tiempo, no pertenecen al puesto ni a la compensación, son fundamentales para análisis de rotación. Por eso se agrupan en una tabla de historial.

### Variables

EmployeeNumber (FK)

PerformanceRating

TotalWorkingYears

YearsAtCompany

YearsInCurrentRole

YearsSinceLastPromotion

YearsWithCurrManager

NumCompaniesWorked

TrainingTimesLastYear

In [26]:
PerformanceHistory = pd.DataFrame(data[[
    'EmployeeNumber',
    'PerformanceRating',
    'TotalWorkingYears',
    'YearsAtCompany',
    'YearsInCurrentRole',
    'YearsSinceLastPromotion',
    'YearsWithCurrManager',
    'NumCompaniesWorked',
    'TrainingTimesLastYear'
]])

PerformanceHistory.head()

,EmployeeNumber,PerformanceRating,TotalWorkingYears,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,NumCompaniesWorked,TrainingTimesLastYear
0,1,3,8,6,4,0,5,8,0
1,2,4,10,10,7,1,7,1,3
2,4,3,7,0,0,0,0,6,3
3,5,3,8,8,7,3,0,1,3
4,7,3,6,2,2,2,2,9,3


## 6. Tabla Attrition

### Justificación

Esta tabla contiene la variable objetivo del proyecto:

- Si el empleado se fue o no.

Separarla tiene ventajas:

- Permite crear vistas limpias

- Permite usarla como etiqueta en modelos ML

- Permite mantener la tabla Employee sin variables de salida

- Facilita análisis de rotación por dimensiones

Es una tabla pequeña pero crucial.

In [27]:
Attrition = pd.DataFrame(data[[
    'EmployeeNumber',
    'Attrition'
]])

Attrition.head()

,EmployeeNumber,Attrition
0,1,Yes
1,2,No
2,4,Yes
3,5,No
4,7,No


----------------------------------------------

# Resumen del Modelo de Base de Datos del Proyecto BI IBM

## Tipo de modelo

El modelo diseñado corresponde a un modelo analítico (OLAP) orientado a Business Intelligence, no a un modelo transaccional (OLTP).
Se basa en una estructura centrada en la entidad Employee, donde todas las tablas se relacionan mediante la clave natural EmployeeNumber.

Este enfoque es común en proyectos de:

- Recursos Humanos

- Rotación de personal

- Evaluación de desempeño

- People Analytics

- Dashboards corporativos

## Justificación del diseño

El dataset original de IBM es un snapshot estático, donde cada fila representa un empleado único y no existen entidades independientes con múltiples registros (como salarios mensuales, evaluaciones periódicas, departamentos con su propio ID, etc.).

### Por ello:

- EmployeeNumber es la única clave natural disponible.

- No existen PK adicionales que puedan definirse sin inventar estructura.

- Las variables se agrupan por naturaleza y función analítica, no por relaciones transaccionales.

- El modelo se organiza en tablas satélite que extienden la información del empleado.

Este diseño corresponde a un Snowflake simplificado centrado en la dimensión Employee, ideal para BI.

---------------------------------------------------

# Exportación de DF´s a .csv para aliemntar posteriormente las tablas de la BBDD

In [28]:
import os

# Verificar en qué carpeta se guardará el archivo
os.getcwd()

# Exportando dataframe´s a CSV´s
Employee.to_csv('Employee.csv', index=False)
JobInfo.to_csv('JobInfo.csv', index=False)
SatisfactionScores.to_csv('SatisfactionScores.csv', index=False)
Compensation.to_csv('Compensation.csv', index=False)
PerformanceHistory.to_csv('PerformanceHistory.csv', index=False)
Attrition.to_csv('Attrition.csv', index=False)